# Fine-Tune Qwen2.5-Coder-1.5B on React Dataset

This notebook fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` on your React component dataset using LoRA adapters.

**Optimized for M2 Mac mini 8GB RAM deployment!**

**Setup Required:**
1. Runtime: GPU (T4, free tier - or better)
2. Google Drive mounted for data and model storage
3. Dataset: `data_cleaned_huggingface_new.jsonl` (39K samples)

**Timeline:**
- Setup: 10 minutes
- Training: 45-60 minutes (1.5B is fast!)
- Model merge: 5 minutes
- **Total: 1-2 hours** (vs 3-4 hours for 7B)

**Model Specifications:**
- Base: Qwen2.5-Coder-1.5B (1.5 billion parameters)
- Output GGUF: ~1.5-2 GB (Q4_K_M quantization)
- Fits on: M2 Mac mini with 8GB RAM ✅

## Check GPU Availability

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ GPU not available! Go to Runtime → Change runtime type and select T4 GPU")

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

## Step 2: Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers peft bitsandbytes accelerate datasets torch
print("✅ Dependencies installed")

## Step 3: Prepare Dataset

Upload `data_cleaned_huggingface_new.jsonl` to your Google Drive first:
1. Create folder in Google Drive: `micro_ai_coder_data`
2. Upload `data_cleaned_huggingface_new.jsonl` there
3. Then run the cell below

In [ ]:
import json
from pathlib import Path

# Path to dataset in Google Drive
dataset_path = "/content/drive/My Drive/micro_ai_coder_data/data_cleaned_huggingface_new.jsonl"

# Check if file exists
if Path(dataset_path).exists():
    # Count samples
    with open(dataset_path) as f:
        samples = [json.loads(line) for line in f if line.strip()]
    print(f"✅ Loaded {len(samples)} samples")
    
    # Show sample
    sample = samples[0]
    print(f"\nSample format:")
    print(f"  Keys: {list(sample.keys())}")
    print(f"  Prompt: {sample['prompt'][:60]}...")
    print(f"  Code length: {len(sample['code'])} chars")
else:
    print(f"❌ Dataset not found at {dataset_path}")
    print(f"\nPlease:")
    print(f"1. Create folder in Google Drive: micro_ai_coder_data")
    print(f"2. Upload data_cleaned_huggingface_new.jsonl there")
    print(f"3. Re-run this cell")

## Step 4: Load Base Model (1.5B - Much Faster!)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
print(f"Loading {model_name}...")
print("This may take 1-2 minutes...\n")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer loaded")

# Configure 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load model in 8-bit to save memory
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print(f"✅ Model loaded (8-bit quantized)")

# Show model info
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params/1e9:.2f}B")
print(f"VRAM requirement: ~3-4 GB (1.5B model is very efficient!)")

## Step 5: Setup LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model

print("Setting up LoRA adapters...\n")

lora_config = LoraConfig(
    r=32,                              # LoRA rank (smaller for 1.5B model)
    lora_alpha=16,                     # LoRA alpha
    target_modules=["q_proj", "v_proj"],  # Which modules to apply LoRA
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Show trainable parameters
model.print_trainable_parameters()
print("\n✅ LoRA adapters applied")
print("Note: Only ~0.2-0.3% of parameters need training!")

## Step 6: Prepare and Tokenize Dataset

In [ ]:
from datasets import Dataset
import json

print("Loading and tokenizing dataset...\n")

# Load data
with open(dataset_path) as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

# Create training examples (prompt + code format)
examples = []
for item in raw_data:
    prompt = item['prompt']
    code = item['code']
    # Format: "Prompt: {prompt}\n\nCode:\n{code}"
    text = f"Prompt: {prompt}\n\nCode:\n{code}"
    examples.append({"text": text})

print(f"Prepared {len(examples)} examples")

# Create HuggingFace dataset
dataset = Dataset.from_dict({"text": [e["text"] for e in examples]})

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,  # Optimal for React components
        padding="max_length"
    )

# Tokenize all samples
print("Tokenizing... (this takes 1-2 minutes)")
tokenized_dataset = dataset.map(tokenize_function, batched=True, batch_size=100, remove_columns=["text"])

# Remove attention_mask for language modeling (not needed for CLM)
tokenized_dataset = tokenized_dataset.remove_columns(["attention_mask"])

# Split: 80% train, 10% val, 10% test using proper dataset methods
split_datasets = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_datasets["train"]
temp_dataset = split_datasets["test"]

# Further split test into val (10%) and test (10%)
split_temp = temp_dataset.train_test_split(test_size=0.5, seed=42)
val_dataset = split_temp["train"]
test_dataset = split_temp["test"]

print(f"\n✅ Dataset split:")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val:   {len(val_dataset)} samples")
print(f"   Test:  {len(test_dataset)} samples")

## Step 7: Configure Training (Optimized for 1.5B)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import os

# Create output directory in Google Drive
output_dir = "/content/drive/My Drive/qwen_1_5b_reactjs_checkpoints"
os.makedirs(output_dir, exist_ok=True)

print("Configuring training for 1.5B model...\n")

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,  # Can use larger batches with 1.5B
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,  # No need for accumulation with 1.5B
    warmup_steps=300,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=2e-4,
    fp16=True,  # Use fp16 for faster training
    gradient_checkpointing=True,  # Save memory
    optim="paged_adamw_32bit",  # Memory-efficient optimizer
)

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("✅ Training configured for 1.5B model")
print(f"   Output dir: {output_dir}")
print(f"   Epochs: 3")
print(f"   Batch size: 8 (efficient for 1.5B)")
print(f"   Learning rate: 2e-4")
print(f"   Optimizer: paged_adamw_32bit")
print(f"   Expected training time: 45-60 minutes")

## Step 8: Start Training ⏱️ (45-60 minutes)

**Monitor:** Watch the loss curves below. Expected behavior:
- Training loss should decrease monotonically
- Validation loss should converge < 1.3
- Fast training with 1.5B model!

**If stuck:** 
- Keep the browser tab open
- Training checkpoints are auto-saved to Google Drive
- With T4 GPU, you have 12+ hours of compute

In [ ]:
print("🚀 Starting fine-tuning of 1.5B model...\n")
print("This will take approximately 45-60 minutes.\n")
print("Expected validation loss: < 1.3\n")

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"\nFinal metrics:")
print(f"  Train loss: {train_result.training_loss:.4f}")

## Step 9: Save Final Model

In [ ]:
print("Saving model...\n")

# Save LoRA adapters
lora_output_dir = f"{output_dir}/lora_adapter"
model.save_pretrained(lora_output_dir)
print(f"✅ LoRA adapters saved to {lora_output_dir}")

# Save tokenizer
tokenizer.save_pretrained(lora_output_dir)
print(f"✅ Tokenizer saved")

## Step 10: Merge LoRA Adapters into Base Model

In [ ]:
print("Merging LoRA adapters with base model...\n")
print("This may take 3-5 minutes...\n")

# Merge adapters
merged_model = model.merge_and_unload()

# Save merged model
merged_output_dir = f"{output_dir}/qwen_1_5b_reactjs_merged"
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)

print(f"\n✅ Merged model saved to {merged_output_dir}")
print(f"\nModel size: ~3-4 GB (fp16)")
print(f"After GGUF Q4_K_M conversion: ~1.5-2 GB")
print(f"\n📝 Next steps:")
print(f"1. Download the merged model from Google Drive")
print(f"2. Place in: finetuned_ai_model_qwen-1-5-b/models/qwen_1_5b_reactjs_merged/")
print(f"3. Run conversion script to create GGUF format")
print(f"4. Deploy as Ollama model on your M2 Mac mini")

## Step 11: Validation (Optional)

Test the fine-tuned model on a few samples

In [ ]:
import torch

print("Testing fine-tuned 1.5B model on sample prompts...\n")

test_prompts = [
    "Write a React button component",
    "Create a React counter with increment/decrement",
    "Generate a React form component"
]

merged_model.eval()
with torch.no_grad():
    for i, prompt in enumerate(test_prompts, 1):
        print(f"Test {i}: {prompt}")
        print("-" * 60)
        
        # Format prompt
        full_prompt = f"Prompt: {prompt}\n\nCode:\n"
        
        # Tokenize
        inputs = tokenizer.encode(full_prompt, return_tensors="pt").to(merged_model.device)
        
        # Generate
        outputs = merged_model.generate(
            inputs,
            max_new_tokens=256,
            temperature=0.3,
            top_k=40,
            top_p=0.9
        )
        
        # Decode
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        code = generated_text.split("Code:\n")[-1].strip()
        
        print(code[:200] + "..." if len(code) > 200 else code)
        print()

## ✅ Complete!

### What's Next:

1. **Download Model** (if not already saved to Drive):
   - Go to Google Drive → `qwen_1_5b_reactjs_checkpoints/qwen_1_5b_reactjs_merged/`
   - Download to local machine
   - Place in: `finetuned_ai_model_qwen-1-5-b/models/qwen_1_5b_reactjs_merged/`

2. **Convert to GGUF** (on local machine):
   - Use `convert_to_gguf.py` script provided
   - Output: `qwen_1_5b_reactjs_merged.Q4_K_M.gguf` (~1.5-2 GB)
   - This fits easily on M2 Mac mini!

3. **Deploy to Ollama** (on local machine):
   - Create Modelfile
   - Register with Ollama: `ollama create micro-ai-coder-1-5b:latest`
   - Test: `ollama run micro-ai-coder-1-5b:latest "Write a React component"`

4. **Update Phase 3/4**:
   - Copy new inference module to main project
   - Copy new agent to main project
   - Run validation tests

### Why 1.5B for M2 Mac mini?

| Aspect | 7B Model | 1.5B Model |
|--------|----------|------------|
| GGUF Size | 4-5 GB | 1.5-2 GB |
| Runtime RAM | 6-8 GB | 2-3 GB |
| Inference Speed | 5-10 tok/s | 15-20 tok/s |
| M2 Fit | Tight | Comfortable |
| Code Quality | Excellent | Very Good |
| Training Time | 2-3 hours | 45-60 min |

---

**Training Status**: ✅ Complete  
**Expected Validation Loss**: < 1.3  
**Expected Success Rate**: ≥90% valid code generation